# The Monty Hall Paradox
The Monty Hall Paradox is a statistical puzzle that is very difficult to fathom. In this notebook, I will attempt to verify its conclusion.

## Setup
The exact setup of this problem is important. There are three doors, two with goats behind them, and one with a car, the grand prize. 

The player's goal in this game is to guess which door hides the car. The player makes a choice, and then the host, Monty, opens one door, a door with a goat behind it. 

After Monty reveals a goat door, the player is asked to choose between sticking with their initial choice, or switching to the remaining unopened door.

## Statistics vs. Intuition
Statistics tells us that there switching is a better strategy than sticking with the first door, *because* a goat door was revealed.

My intuition tells me that the revelation of information should not affect the physical reality revealed at the very end.

## Experiment
By recreating the Monty Hall setup, and recording how often switching and not switching initial choices after initial choice and revelation lead to a correct guess, and running the experiment several times, we can investigate whether statistics or intuition are right about this one. 

### A Result Type
We define a `Result` type that holds the counts of initial guesses and switched guesses winning.

In [13]:
Results = class Results {
  initial: Number = 0;
  switched: Number = 0;
  total: Number = 0;

  get pcInitial(){
    return this.initial / this.correct;
  }

  get pcSwitched(){
    return this.switched / this.correct;
  }

  get correct(){
    return this.initial + this.switched;
  }

  get pcCorrect(){
    return this.correct / this.total;
  }
  
  toString(){
    const html = `
    <tr>
      <th>Initial Guess Correct</th><td>${this.initial}</td><td>(${this.pcInitial.toFixed(2).substring(2)}%)</td>
    </tr>
    <tr>
      <th>Switched Guess Correct</th><td>${this.switched}</td><td>(${this.pcSwitched.toFixed(2).substring(2)}%)</td>
    </tr>
    <tr>
      <th>Total Correct</th><td>${this.correct}</td><td>(${this.pcCorrect.toFixed(2).substring(2)}%)</td>
    </tr>
    <tr>
      <th>Total Incorrect</th><td>${this.total - this.correct}</td><td>(${(1.0 - this.pcCorrect).toFixed(2).substring(2)}%)</td>
    </tr>
    <tr>
      <th>Total</th><td>${this.total}</td>
    <tr>
    `

    const table = document.createElement('table');
    table.innerHTML = html;
    return table;
  }
}

In [9]:
return new Results()

<pre>{
  "initial": 0,
  "switched": 0,
  "total": 0
}</pre>

### A Choose Function
The `choose` function accepts a list, and returns the **index** of a chosen element.

In [9]:
choose = function choose(prizeList: Array<String>): Number {
  return Math.floor(Math.random() * prizeList.length);
}

In [9]:
return choose(["goat", "goat", "car"])

2

### A Reveal Function
The `reveal` function returns the **index** of a not previously chosen goat door.

In [9]:
reveal = function reveal(
  prizes: Array<String>, 
  prize: String,
  initialChoice: Number
): Number {
  const prizeList = [...prizes];
  const prizeIndex = prizeList.indexOf(prize);
  const eligiblePrizeIndices = prizeList
    .map((_: String, index: Number): Array<Number?> => {
      if(index === initialChoice) return null;
      if(index === prizeIndex) return null;
      return index;
    })
    .filter((index: Number?): Boolean => {
      return index !== null;
    });

  const revealed = choose(eligiblePrizeIndices);
  return eligiblePrizeIndices[revealed];
}

Since the initial list is 3 elements long, and after choosing a door at least one but at most two elements remain for the host to reveal, the output of this function should only ever be 1 or 2.

In [9]:
const prizeList = ["car", "goat", "goat"];
return reveal(prizeList, "car", choose(prizeList));

1

### A Shuffle Function
The `shuffle` function constructs a shuffled array from an input array.

In [9]:
shuffle = function shuffle(inputList: Array<string>): Array<string> {
  const sourceList: Array<string> = [...inputList]
  const shuffledList: Array<string> = [];
  while(sourceList.length){
    shuffledList.push(sourceList.splice(Math.floor(Math.random() * sourceList.length), 1);
  }

  return shuffledList;
}

In [9]:
return shuffle(["car", "goat", "goat"])

<pre>[
  [
    "goat"
  ],
  [
    "car"
  ],
  [
    "goat"
  ]
]</pre>

### The Experiment Function
The experiment function accepts a list of prizes representing the 3 doors, an object for storing the results (defined above), shuffles the list, chooses an initial guess, chooses a revelation from the remaining eligible doors, and incrememnts the result object's `initial` or `switched` elements, depending on which result was correct.

In [9]:
prizes = ["car", "goat", "goat"];
results = new Results();

In [9]:
experiment = function experiment(prizes: Array<String>, mainPrize: String, results: Results){
  const doors = shuffle([...prizes]);

  const prize = doors.findIndex(p => p == mainPrize);
  const firstGuess = choose(doors);
  const revealed = reveal(doors, mainPrize, firstGuess);

  const remainingDoors = doors.filter((_, index) => {
    return (index !== revealed);
  })
  const secondGuess = choose(remainingDoors);

  results.initial += ((firstGuess === prize) && (secondGuess === firstGuess)) ? 1 : 0;
  results.switched += ((secondGuess === prize) && (secondGuess !== firstGuess)) ? 1 : 0;
  results.total += 1;
}

With the experiment and its auxiliary functions defined, click **Run All** (at the very bottom) once, and the **Run** button (in the cell after this one) repeatedly.

In [184]:
for(let i=0; i<100; i++){
  experiment(prizes, "car", results);
}
return results.toString();

<table>
    <tbody><tr>
      <th>Initial Guess Correct</th><td>421</td><td>(32%)</td>
    </tr>
    <tr>
      <th>Switched Guess Correct</th><td>890</td><td>(68%)</td>
    </tr>
    <tr>
      <th>Total Correct</th><td>1311</td><td>(33%)</td>
    </tr>
    <tr>
      <th>Total Incorrect</th><td>2689</td><td>(67%)</td>
    </tr>
    <tr>
      <th>Total</th><td>4000</td>
    </tr><tr>
    </tr></tbody></table>

## Analysis
This setup compares the number of correct first guesses to the number of correct switched guesses. These results show values of **32% for the initial guesses** being correct, and **68% for switched guesses** being correct. I expected values of 30% and 50%, respectively, but even though this experiment deviates from those numbers, it consistently establishes the switched guess to be correct more often than the initial guess, after a goat door was revealed.